# HACKOWEEK SEM 5 - Week 5 & Week 6
## Mathematics for Machine Learning: Linear Algebra & Calculus

**Student:** Samruddhi Kalbande  
**Course:** B.Tech Computer Science / Information Technology (5th Semester)  
**Topics Covered:**
1. **Linear Algebra:** Vectors, Matrices, Dot Products, and Eigenvalues/Eigenvectors (Intuition-level)
2. **Calculus:** Derivatives, Gradients, and the Chain Rule (Backpropagation intuition)
3. **Applied ML Applications:** Covariance Eigen-decomposition (PCA) and Gradient Descent Regression on the Kaggle Student Performance Dataset

### 1. Setup & Environment
Importing NumPy for matrix calculations, Pandas for loading dataset, and Matplotlib for visualizing vector fields and loss curves.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load student dataset
df = pd.read_csv('../data/kaggle_student_performance.csv')
print(f"Loaded {len(df)} student records for mathematical analysis.")
df[['StudentID', 'StudyHoursPerWeek', 'CGPA', 'MathScore', 'ReadingScore', 'WritingScore']].head(3)

---
## 2. Linear Algebra: Vectors, Norms & Dot Products
A vector $\mathbf{u} \in \mathbb{R}^n$ represents a direction and magnitude in feature space.
- **Euclidean Norm ($L_2$):** $\|\mathbf{u}\| = \sqrt{\sum_{i=1}^n u_i^2}$
- **Dot Product:** $\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^n u_i v_i = \|\mathbf{u}\| \|\mathbf{v}\| \cos(\theta)$
- **Cosine Similarity:** $\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$

In [2]:
# Example: Student Academic Vector Comparison (Math, Reading, Writing)
student_A = df.loc[0, ['MathScore', 'ReadingScore', 'WritingScore']].to_numpy(dtype=float)
student_B = df.loc[1, ['MathScore', 'ReadingScore', 'WritingScore']].to_numpy(dtype=float)

norm_A = np.linalg.norm(student_A)
norm_B = np.linalg.norm(student_B)
dot_prod = np.dot(student_A, student_B)
cos_sim = dot_prod / (norm_A * norm_B)
angle_deg = np.degrees(np.arccos(cos_sim))

print(f"Student A Vector: {student_A} | Norm: {norm_A:.2f}")
print(f"Student B Vector: {student_B} | Norm: {norm_B:.2f}")
print(f"Dot Product: {dot_prod:.2f}")
print(f"Cosine Similarity: {cos_sim:.4f} (Angle: {angle_deg:.2f}°)")

---
## 3. Matrices as Linear Transformations
A 2D matrix $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$ maps standard basis vectors $\mathbf{i} = [1, 0]^T$ and $\mathbf{j} = [0, 1]^T$ to transformed space.
- **Determinant $\det(A)$:** Represents the factor by which area is scaled under transformation.

In [3]:
# Define 2D rotation matrix of 45 degrees + 1.5x scaling
theta = np.radians(45)
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
]) * 1.5

# Unit square vertices
square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])
transformed_square = R @ square

plt.figure(figsize=(6, 6))
plt.plot(square[0], square[1], 'b--', label='Original Unit Square')
plt.plot(transformed_square[0], transformed_square[1], 'r-', linewidth=2, label='Transformed (Rotated + Scaled)')
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.grid(True, linestyle=':')
plt.title(f'2D Matrix Linear Transformation (Det = {np.linalg.det(R):.2f})')
plt.legend()
plt.show()

---
## 4. Eigenvalues & Eigenvectors (Intuition-Level & PCA)
An eigenvector $\mathbf{v}$ of matrix $A$ maintains its directional alignment under transformation, scaling only by its eigenvalue $\lambda$:
$$A \mathbf{v} = \lambda \mathbf{v}$$

In Machine Learning, computing eigenvectors of a **Covariance Matrix** finds the **Principal Components (PCA)** — the orthogonal axes of maximum variance.

In [4]:
# Extract exam scores matrix
X = df[['MathScore', 'ReadingScore', 'WritingScore']].to_numpy()
X_centered = X - np.mean(X, axis=0)

# Compute Covariance Matrix: Sigma = (1 / (N - 1)) * X^T * X
cov_matrix = np.cov(X_centered, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

var_ratios = (eigenvalues / np.sum(eigenvalues)) * 100

print("Covariance Matrix (Math, Reading, Writing):\n", np.round(cov_matrix, 2))
print("\nEigenvalues (Variance along Principal Axes):", np.round(eigenvalues, 3))
print("Explained Variance Ratio (%):", np.round(var_ratios, 1))
print("\nPrimary Eigenvector (PC1 Direction):", np.round(eigenvectors[:, 0], 3))
print(f"Conclusion: PC1 captures {var_ratios[0]:.1f}% of total performance variance across subjects!")

---
## 5. Calculus: Derivatives, Gradients & Gradient Descent
A **gradient** $\nabla J(\mathbf{w})$ points in the direction of steepest increase of cost function $J$:
$$\nabla J(w, b) = \begin{bmatrix} \frac{\partial J}{\partial w} \\ \frac{\partial J}{\partial b} \end{bmatrix}$$

To minimize Mean Squared Error $J(w, b) = \frac{1}{2m} \sum_{i=1}^m (\hat{y}_i - y_i)^2$, we update parameters in the opposite direction:
$$w := w - \alpha \frac{\partial J}{\partial w}, \quad b := b - \alpha \frac{\partial J}{\partial b}$$

In [5]:
# Fitting Study Hours vs CGPA using Gradient Descent
X_hrs = df['StudyHoursPerWeek'].to_numpy()
y_cgpa = df['CGPA'].to_numpy()
m = len(X_hrs)

# Standardize input for fast convergence
x_mean, x_std = np.mean(X_hrs), np.std(X_hrs)
X_scaled = (X_hrs - x_mean) / x_std

w = 0.0
b = np.mean(y_cgpa)
alpha = 0.05
epochs = 60
loss_history = []

for epoch in range(epochs):
    y_pred = w * X_scaled + b
    loss = (1 / (2 * m)) * np.sum((y_pred - y_cgpa) ** 2)
    loss_history.append(loss)

    # Partial derivatives
    dw = (1 / m) * np.sum((y_pred - y_cgpa) * X_scaled)
    db = (1 / m) * np.sum(y_pred - y_cgpa)

    # Gradient step
    w -= alpha * dw
    b -= alpha * db

w_orig = w / x_std
b_orig = b - (w * x_mean / x_std)

print(f"Gradient Descent Converged after {epochs} epochs.")
print(f"Final Equation: CGPA = {w_orig:.3f} * StudyHours + {b_orig:.2f}")
print(f"Initial Loss: {loss_history[0]:.4f} -> Final Loss: {loss_history[-1]:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='purple', linewidth=2)
plt.title('Gradient Descent Loss Curve (MSE Convergence)')
plt.xlabel('Epoch')
plt.ylabel('Loss J(w, b)')
plt.grid(True, linestyle=':')
plt.show()

---
## 6. Calculus: Chain Rule & Backpropagation Intuition
Backpropagation computes the gradient of the loss with respect to every weight in a neural network via the **Chain Rule**:
$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2} \cdot \frac{\partial z_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial W_1}$$

Let's trace a forward pass and backward pass step-by-step.

In [6]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Sample single neuron connection
x_val = 2.5
target = 1.0
w1, b1 = 0.6, 0.2
w2, b2 = 0.9, -0.3

# 1. FORWARD PASS
z1 = w1 * x_val + b1
a1 = sigmoid(z1)
z2 = w2 * a1 + b2
y_hat = sigmoid(z2)
loss = 0.5 * (target - y_hat) ** 2

# 2. BACKWARD PASS (CHAIN RULE)
dL_dyhat = -(target - y_hat)
dyhat_dz2 = y_hat * (1.0 - y_hat)
dL_dz2 = dL_dyhat * dyhat_dz2

dL_dw2 = dL_dz2 * a1
dL_db2 = dL_dz2 * 1.0

dL_da1 = dL_dz2 * w2
da1_dz1 = a1 * (1.0 - a1)
dL_dz1 = dL_da1 * da1_dz1
dL_dw1 = dL_dz1 * x_val

print("--- Forward Pass ---")
print(f"z1: {z1:.3f} -> a1: {a1:.3f} -> z2: {z2:.3f} -> Prediction: {y_hat:.3f}")
print(f"Target: {target} | Error Loss: {loss:.5f}")

print("\n--- Backward Pass (Chain Rule Gradients) ---")
print(f"dL / dw2: {dL_dw2:.6f}")
print(f"dL / dw1: {dL_dw1:.6f}")
print("\nBoth weights receive negative gradients, indicating an upward update will reduce prediction error!")